In [1]:
import pandas as pd

In [4]:
# ============================================================
# 1. Cargar bases
# ============================================================

res = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/resultados_distrital.xlsx")           # resultados 2018–2022
scr_2018 = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scraper_alcaldes_2018.xlsx")     # candidatos 2018
scr_2022 = pd.read_excel("/Users/karlavega/Proyectos/Colaborativos/Forest_Peru/Datos/BD/scraper_alcaldes_2022.xlsx")     # candidatos 2022


In [6]:
# ============================================================
# 2. Normalizar columnas clave
# ============================================================

def limpiar_ubigeo(s):
    return (s.astype(str)
              .str.strip()
              .str.replace(r"\.0$", "", regex=True)  # por si vienen como float
              .str.zfill(5))                         # ajusta a 5 o 6 según tu estándar

for df in [res, scr_2018, scr_2022]:
    df["ubigeo"] = limpiar_ubigeo(df["ubigeo"])
    df["organizacion_politica"] = (
        df["organizacion_politica"]
          .astype(str)
          .str.strip()
          .str.upper()
    )


In [7]:
# ============================================================
# 3. Quedarse solo con ALCALDE DISTRITAL en los scrapers
#    y dejar una sola fila por ubigeo–partido (candidato único)
# ============================================================

def preparar_scraper(df_scr, anio):
    df = df_scr.copy()
    df = df[df["cargo"].str.upper().str.contains("ALCALDE DISTRITAL")]

    # Nos quedamos con una fila por ubigeo–partido (por si hay duplicados)
    claves = ["ubigeo", "organizacion_politica"]
    cols_candidato = claves + [
        "departamento", "provincia", "distrito",
        "jurado_electoral", "dni", "nombres", "apellidos"
    ]
    cols_candidato = [c for c in cols_candidato if c in df.columns]

    df = (
        df[cols_candidato]
        .drop_duplicates(subset=claves)
        .reset_index(drop=True)
    )

    df["año"] = anio
    return df

scr_2018_clean = preparar_scraper(scr_2018, 2018)
scr_2022_clean = preparar_scraper(scr_2022, 2022)



In [8]:
# ============================================================
# 4. Separar resultados por año y unir con su scraper correspondiente
# ============================================================

res_2018 = res[res["año"] == 2018].copy()
res_2022 = res[res["año"] == 2022].copy()

# Claves de unión: ubigeo + partido + año
claves_merge = ["ubigeo", "organizacion_politica", "año"]

# Para 2018
base_2018 = res_2018.merge(
    scr_2018_clean,
    on=["ubigeo", "organizacion_politica", "año"],
    how="left",
    suffixes=("", "_scraper")
)

# Para 2022
base_2022 = res_2022.merge(
    scr_2022_clean,
    on=["ubigeo", "organizacion_politica", "año"],
    how="left",
    suffixes=("", "_scraper")
)


In [9]:
# ============================================================
# 5. Unir en una sola base 2018 + 2022
# ============================================================

base_final = pd.concat([base_2018, base_2022], ignore_index=True)

# Opcional: ordenar columnas
cols_primero = [
    "año", "ubigeo", "region", "provincia", "distrito",
    "organizacion_politica", "total_votos", "orden_aparicion",
    "nombres", "apellidos", "dni", "cargo"
]
cols_primero = [c for c in cols_primero if c in base_final.columns]
otras = [c for c in base_final.columns if c not in cols_primero]
base_final = base_final[cols_primero + otras]


In [10]:
# ============================================================
# 6. Guardar a Excel / CSV
# ============================================================

base_final.to_excel("base_distrital_2018_2022_con_candidatos.xlsx", index=False)
base_final.to_csv("base_distrital_2018_2022_con_candidatos.csv", index=False, encoding="utf-8-sig")

print("Listo. Filas en la base final:", len(base_final))

Listo. Filas en la base final: 20694
